In [1]:
import pandas as pd
import numpy as np
from mlforecast import MLForecast
import sys
import os
sys.path.append(os.path.abspath("../.."))
from mlforecast.lag_transforms import RollingMean, RollingStd
import lightgbm as lgb
from tinyshift.modelling import FirstStageForecasterEvaluator, TwoStageForecasterEvaluator, TwoStageForecasterWrapper


In [2]:
def generate_m5_simulated_data(n_stores=3, n_skus=5, start_date="2023-01-01", days=365, seed=42):

    np.random.seed(seed)
    dates = pd.date_range(start=start_date, periods=days, freq="D")
    n_days = len(dates)
    
    data_list = []

    # Calendar signals are shared by the whole retail panel, as in M5.
    event_mask = np.zeros(n_days, dtype=bool)
    event_mask[np.random.choice(n_days, size=min(12, n_days), replace=False)] = True
    holiday_anchor = pd.Series(dates.strftime('%m-%d').isin(['01-01', '07-04', '11-24', '12-25']))
    holiday_window = (holiday_anchor | holiday_anchor.shift(-1, fill_value=False)
                      | holiday_anchor.shift(-2, fill_value=False)).to_numpy()

    for store_id in range(1, n_stores + 1):
        store_name = f"STORE_{store_id:02d}"
        for sku_id in range(1, n_skus + 1):
            item_id = f"FOODS_1_{sku_id:03d}"
            
            regime = ['smooth', 'intermittent', 'lumpy'][(sku_id - 1) % 3]
            if regime == 'smooth':
                base_demand = np.random.uniform(8.0, 16.0)
                dispersion = np.random.uniform(35.0, 60.0)
            elif regime == 'intermittent':
                base_demand = np.random.uniform(0.5, 1.8)
                dispersion = np.random.uniform(4.0, 8.0)
            else:
                base_demand = np.random.uniform(4.0, 9.0)
                dispersion = np.random.uniform(2.5, 5.0)
            
            store_factor = 0.90 + 0.10 * store_id
            dow_factor = np.tile([0.78, 0.82, 0.88, 0.95, 1.08, 1.32, 1.22], int(np.ceil(n_days/7)))[:n_days]
            trend_rate = np.random.uniform(-0.08, 0.18)
            trend_factor = np.exp(trend_rate * np.arange(n_days) / 365.25)
            annual_factor = 1.0 + 0.12 * np.sin(2 * np.pi * (dates.dayofyear.to_numpy() - 30) / 365.25)
            
            base_price = np.random.uniform(2.0, 15.0)
            prices = base_price * (1.0 + 0.02 * np.arange(n_days) / 365.25)
            promo_mask = np.zeros(n_days, dtype=bool)
            eligible_starts = np.arange(7, max(8, n_days - 7))
            promo_starts = np.random.choice(eligible_starts, size=min(10, len(eligible_starts)), replace=False)
            for promo_start in promo_starts:
                promo_duration = np.random.randint(3, 8)
                promo_mask[promo_start:promo_start + promo_duration] = True
            promo_discount = np.random.choice([0.70, 0.85])
            prices[promo_mask] *= promo_discount
            price_elasticity = np.exp(-0.15 * (prices - base_price))
            
            event_impact = np.ones(n_days)
            event_impact[event_mask] = np.random.uniform(1.2, 1.8, size=event_mask.sum())
            
            holiday_factor = np.where(holiday_window, 1.25, 1.0)
            lambda_t = (base_demand * store_factor * dow_factor * trend_factor
                        * annual_factor * price_elasticity * event_impact * holiday_factor)
            
            probability = dispersion / (dispersion + lambda_t)
            sales = np.random.negative_binomial(dispersion, probability)
            
            for d_idx, d_date in enumerate(dates):
                data_list.append({
                    'date': d_date,
                    'store_id': store_name,
                    'item_id': item_id,
                    'sales': sales[d_idx],
                    'true_lambda': lambda_t[d_idx],
                    'sell_price': round(prices[d_idx], 2),
                    'is_event': int(event_mask[d_idx]),
                    'is_holiday_window': int(holiday_window[d_idx]),
                    'is_promo': int(promo_mask[d_idx]),
                    'demand_regime': regime,
                    'true_dispersion': dispersion,
                })

    return pd.DataFrame(data_list)

df_raw = generate_m5_simulated_data(n_stores=3, n_skus=5, days=365)

df_raw['unique_id'] = df_raw['store_id'] + '_' + df_raw['item_id']

df_nixtla = df_raw.rename(columns={
    'date': 'ds',
    'sales': 'y'
})

In [3]:
def temporal_split_by_horizon(df, time_col='ds', horizon_days=28):
    df = df.sort_values(time_col)
    max_date = df[time_col].max()
    cutoff_date = max_date - pd.Timedelta(days=horizon_days)
    
    df_train = df[df[time_col] <= cutoff_date].copy()
    df_test = df[df[time_col] > cutoff_date].copy()
    
    return df_train, df_test, cutoff_date

In [4]:
def add_relative_price(df: pd.DataFrame, reference_df: pd.DataFrame, id_col: str = 'unique_id', price_col: str = 'sell_price') -> pd.DataFrame:
    df = df.copy()
    mean_price_per_sku = reference_df.groupby(id_col)[price_col].mean()
    df['relative_price'] = df[price_col] / (df[id_col].map(mean_price_per_sku) + 1e-6)
    
    return df

df_train, df_test, cutoff = temporal_split_by_horizon(df_nixtla, horizon_days=28)
df_train = add_relative_price(df_train, reference_df=df_train)
df_test = add_relative_price(df_test, reference_df=df_train)

In [5]:
fcst = MLForecast(
    models={
        'tsf': lgb.LGBMRegressor(
            objective='poisson',
            metric='poisson',
            n_estimators=500,
            learning_rate=0.03,
            random_state=42,
            verbosity=-1,
        )
    },
    freq='D',
    lags=[7, 14, 28],
    lag_transforms={
        1: [RollingMean(window_size=7), RollingStd(window_size=7)],
    },
    date_features=['dayofweek', 'month', 'dayofyear']
)

In [6]:
tsf = TwoStageForecasterWrapper(fcst)
tsf.fit(df_train[["ds", "y", "sell_price", 'relative_price', "is_event", "is_holiday_window", "unique_id"]], h=28, n_windows=10, refit=True, gamma=None)

,fcst,MLForecast(mo...num_threads=1)


In [7]:
df_test[["ds", "sell_price", "is_event", "is_holiday_window", "unique_id"]].groupby("unique_id")["ds"].nunique()

unique_id
STORE_01_FOODS_1_001    28
STORE_01_FOODS_1_002    28
STORE_01_FOODS_1_003    28
STORE_01_FOODS_1_004    28
STORE_01_FOODS_1_005    28
STORE_02_FOODS_1_001    28
STORE_02_FOODS_1_002    28
STORE_02_FOODS_1_003    28
STORE_02_FOODS_1_004    28
STORE_02_FOODS_1_005    28
STORE_03_FOODS_1_001    28
STORE_03_FOODS_1_002    28
STORE_03_FOODS_1_003    28
STORE_03_FOODS_1_004    28
STORE_03_FOODS_1_005    28
Name: ds, dtype: int64

In [8]:
df_res = tsf.pmf(h=28, X_df=df_test[["ds", "sell_price", "relative_price", "is_event", "is_holiday_window", "unique_id"]], max_k=4)
df_res = df_res.merge(
    df_test[["unique_id", "ds", "y", "true_lambda", "demand_regime"]],
    on=["unique_id", "ds"],
    how="left",
    validate="one_to_one",
)

In [9]:
df_res

,unique_id,ds,lambda_t,r_dispersion,P(Y=0),P(Y=1),P(Y=2),P(Y=3),P(Y=4),P(Y>4),y,true_lambda,demand_regime
0,STORE_01_FOODS_1_001,2023-12-04,10.291950,17.436385,0.000307,0.001987,0.006798,0.016348,0.031001,0.943559,5,8.902477,smooth
1,STORE_01_FOODS_1_001,2023-12-05,15.733804,17.436385,0.000013,0.000112,0.000488,0.001500,0.003635,0.994253,10,14.643827,smooth
2,STORE_01_FOODS_1_001,2023-12-06,9.591885,17.436385,0.000479,0.002967,0.009706,0.022317,0.040463,0.924067,14,10.335981,smooth
3,STORE_01_FOODS_1_001,2023-12-07,13.038693,17.436385,0.000059,0.000441,0.001740,0.004823,0.010543,0.982393,10,11.763518,smooth
4,STORE_01_FOODS_1_001,2023-12-08,15.711714,17.436385,0.000014,0.000113,0.000493,0.001514,0.003666,0.994200,14,14.394138,smooth
...,...,...,...,...,...,...,...,...,...,...,...,...,...
415,STORE_03_FOODS_1_005,2023-12-27,1.694286,4.764879,0.234662,0.293295,0.221756,0.131167,0.066790,0.052330,2,1.894943,intermittent
416,STORE_03_FOODS_1_005,2023-12-28,2.293649,4.764879,0.153750,0.238056,0.222973,0.163382,0.103060,0.118778,2,2.158268,intermittent
417,STORE_03_FOODS_1_005,2023-12-29,2.642456,4.764879,0.122180,0.207682,0.213552,0.171787,0.118962,0.165836,4,2.642847,intermittent
418,STORE_03_FOODS_1_005,2023-12-30,2.618410,4.764879,0.124088,0.209686,0.214347,0.171413,0.118007,0.162460,3,2.447268,intermittent


In [10]:
df_res = tsf.marginal_benefit(h=28, X_df=df_test[["ds", "sell_price", "relative_price", "is_event", "is_holiday_window", "unique_id"]], underage_cost=200, overage_cost=100, max_k=4)
df_res = df_res.merge(
    df_test[["unique_id", "ds", "y", "true_lambda", "demand_regime"]],
    on=["unique_id", "ds"],
    how="left",
    validate="one_to_one",
)
df_res

,unique_id,ds,lambda_t,r_dispersion,MB(k=0),MB(k=1),MB(k=2),MB(k=3),MB(k=4),y,true_lambda,demand_regime
0,STORE_01_FOODS_1_001,2023-12-04,10.291950,17.436385,200.0,199.907900,199.311841,197.272410,192.368108,5,8.902477,smooth
1,STORE_01_FOODS_1_001,2023-12-05,15.733804,17.436385,200.0,199.995952,199.962468,199.816062,199.366136,10,14.643827,smooth
2,STORE_01_FOODS_1_001,2023-12-06,9.591885,17.436385,200.0,199.856153,198.966048,196.054170,189.359130,14,10.335981,smooth
3,STORE_01_FOODS_1_001,2023-12-07,13.038693,17.436385,200.0,199.982258,199.849901,199.327888,197.880901,10,11.763518,smooth
4,STORE_01_FOODS_1_001,2023-12-08,15.711714,17.436385,200.0,199.995904,199.962055,199.814156,199.359981,14,14.394138,smooth
...,...,...,...,...,...,...,...,...,...,...,...,...
415,STORE_03_FOODS_1_005,2023-12-27,1.694286,4.764879,200.0,129.601431,41.612884,-24.913940,-64.264054,2,1.894943,intermittent
416,STORE_03_FOODS_1_005,2023-12-28,2.293649,4.764879,200.0,153.875005,82.458107,15.566144,-33.448458,2,2.158268,intermittent
417,STORE_03_FOODS_1_005,2023-12-29,2.642456,4.764879,200.0,163.345924,101.041349,36.975600,-14.560371,4,2.642847,intermittent
418,STORE_03_FOODS_1_005,2023-12-30,2.618410,4.764879,200.0,162.773607,99.867823,35.563744,-15.860158,3,2.447268,intermittent


In [19]:
df_res = tsf.predict(h=28, X_df=df_test[["ds", "sell_price", "relative_price", "is_event", "is_holiday_window", "unique_id"]], quantiles=(0.05, 0.50, 0.95, 0.99))
df_res = df_res.merge(
    df_test[["unique_id", "ds", "y", "true_lambda", "demand_regime"]],
    on=["unique_id", "ds"],
    how="left",
    validate="one_to_one",
)

In [20]:
# The evaluator expects genuinely out-of-sample predictions. df_res uses the held-out test period.
first_stage_summary = FirstStageForecasterEvaluator.evaluate(
    df_res,
    id_col="unique_id",
)
first_stage_summary

,Metrics
WAPE,39.2560
PBias,1.3700
Score,40.6273
Forecast Instability,18.3592
False Demand on Zero-Days (Avg Pred),1.8275
Peak Demand Deviation (%),-2.2100


### First-stage conditional-mean diagnostics

In [21]:
calibration = FirstStageForecasterEvaluator.calibration_table(
    df_res,
    n_bins=10,
)
calibration

,Calibration Bin,Count,Mean_Prediction,Mean_Observed,Mean_Residual
0,"(0.755, 1.219]",42,0.991996,0.857143,-0.134853
1,"(1.219, 1.634]",42,1.380553,1.095238,-0.285315
2,"(1.634, 2.131]",42,1.857849,1.785714,-0.072135
3,"(2.131, 3.943]",42,2.630069,2.476190,-0.153879
4,"(3.943, 7.253]",42,5.588187,6.238095,0.649909
5,"(7.253, 8.852]",42,8.055833,7.190476,-0.865357
6,"(8.852, 10.154]",42,9.482585,10.976190,1.493606
7,"(10.154, 11.825]",42,10.821473,10.619048,-0.202426
8,"(11.825, 14.916]",42,13.504096,11.666667,-1.837429
9,"(14.916, 25.059]",42,17.081763,17.523810,0.442047


In a calibrated model, `Mean_Observed` is close to `Mean_Prediction` and `Mean_Residual` is close to zero in every bin. Persistent positive residuals indicate underforecasting; negative residuals indicate overforecasting.

In [22]:
TwoStageForecasterEvaluator.evaluate(df_res, quantiles=(0.05, 0.50, 0.95, 0.99))

,Pinball Loss,Target Coverage,Empirical Coverage,Coverage Gap
q_5,0.2589,0.05,0.1595,0.1095
q_50,1.3440,0.50,0.5714,0.0714
q_95,0.4417,0.95,0.9548,0.0048
q_99,0.1197,0.99,0.9929,0.0029


In [39]:
import joblib

model_path = "tsf.joblib"

# Save the fitted wrapper (works the same for mode="local" or mode="global").
joblib.dump(tsf, model_path)

# Load it back into a new object.
loaded_forecast = joblib.load(model_path)

In [40]:
loaded_forecast.predict(h=12, X_df=df_test[["ds", "sell_price", "relative_price", "is_event", "is_holiday_window", "unique_id"]])

,unique_id,ds,lambda_t,r_dispersion,q_5,q_50,q_95
0,STORE_01_FOODS_1_001,2023-12-04,10.291950,17.436385,4,10,17
1,STORE_01_FOODS_1_001,2023-12-05,15.733804,17.436385,8,15,25
2,STORE_01_FOODS_1_001,2023-12-06,9.591885,17.436385,4,9,16
3,STORE_01_FOODS_1_001,2023-12-07,13.038693,17.436385,6,13,22
4,STORE_01_FOODS_1_001,2023-12-08,15.711714,17.436385,8,15,25
...,...,...,...,...,...,...,...
175,STORE_03_FOODS_1_005,2023-12-11,1.841669,4.764879,0,2,5
176,STORE_03_FOODS_1_005,2023-12-12,1.805410,4.764879,0,2,5
177,STORE_03_FOODS_1_005,2023-12-13,1.784450,4.764879,0,1,5
178,STORE_03_FOODS_1_005,2023-12-14,2.199542,4.764879,0,2,6
